# Komp. A — Region map (Inntal / Steinberge)

Visualises the refined region polygon over an OpenTopoMap basemap,
together with:

- the **seed bbox** (red dashed) — what the placeholder config used
- the **selected HydroBASINS Level 8 basins** (yellow fill) — before clipping
- the **refined polygon** (green fill) — basins ∩ seed bbox, after the
  north-of-Inn hauptkamm filter
- the **Inn-line approximation** (blue dashed) used by the hauptkamm filter
- anchor cities (Innsbruck, Schwaz, Wörgl, Kufstein)

Map saved to `data/regions/inntal_steinberge_map.png`. Run after
`python -m alptherm_icon.regions refine-region inntal_steinberge`.

In [ ]:
from pathlib import Path

import contextily as cx
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import LineString, box

from alptherm_icon.regions.basins import INN_LINE, select_basins
from alptherm_icon.regions.polygon import load_region

REGION = "inntal_steinberge"
PROJECT_ROOT = Path.cwd().resolve().parents[0]
CONFIG_PATH = PROJECT_ROOT / "configs" / "regions" / f"{REGION}.geojson"
BASINS_SHP = PROJECT_ROOT / "data" / "basins" / "hybas_eu_lev08_v1c.shp"

geom, props = load_region(CONFIG_PATH, name=REGION)
seed_bounds = tuple(props.get("seed_bounds", geom.bounds))
seed_bbox = box(*seed_bounds)
print(f"refined area:  {props.get('n_basins')} basins, status={props['status']}")
print(f"seed bounds:   {seed_bounds}")

In [ ]:
# Load full HydroBASINS L8 once, re-derive the selected basins for plotting
# (the refined GeoJSON only stores the unioned polygon, not the individual basins).
basins = gpd.read_file(BASINS_SHP)
selected = select_basins(basins, seed_bbox)
print(f"selected {len(selected)} basins: {selected['HYBAS_ID'].tolist()}")

# Inn-line segment across the seed bbox
a, b = INN_LINE
x0, _, x1, _ = seed_bounds
inn_line = LineString([(x0, a + b * x0), (x1, a + b * x1)])

cities = {
    "Innsbruck": (11.394, 47.265),
    "Schwaz": (11.708, 47.343),
    "Wörgl": (12.077, 47.485),
    "Kufstein": (12.171, 47.583),
}

# Reproject everything to Web Mercator (EPSG:3857) for contextily basemap.
refined_gdf = gpd.GeoDataFrame(geometry=[geom], crs="EPSG:4326").to_crs(3857)
seed_gdf = gpd.GeoDataFrame(geometry=[seed_bbox], crs="EPSG:4326").to_crs(3857)
basins_3857 = selected.to_crs(3857)
inn_gdf = gpd.GeoDataFrame(geometry=[inn_line], crs="EPSG:4326").to_crs(3857)
cities_gdf = gpd.GeoDataFrame(
    {"name": list(cities.keys())},
    geometry=gpd.points_from_xy([c[0] for c in cities.values()], [c[1] for c in cities.values()]),
    crs="EPSG:4326",
).to_crs(3857)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))

# Layer order (bottom -> top): basins, refined, seed-bbox outline, Inn line, cities.
basins_3857.plot(ax=ax, facecolor="#f1c40f", edgecolor="#b7950b", alpha=0.25, linewidth=1.0)
refined_gdf.plot(ax=ax, facecolor="#27ae60", edgecolor="#1e8449", alpha=0.45, linewidth=1.5)
seed_gdf.boundary.plot(ax=ax, color="#c0392b", linewidth=1.2, linestyle="--")
inn_gdf.plot(ax=ax, color="#2471a3", linewidth=1.5, linestyle="--")

for _, row in cities_gdf.iterrows():
    ax.plot(row.geometry.x, row.geometry.y, marker="o", color="black", markersize=4)
    ax.annotate(
        row["name"],
        xy=(row.geometry.x, row.geometry.y),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=9,
        color="black",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7),
    )

# Frame the view on the seed bbox with a small margin so the basins
# extending outside are visible.
minx, miny, maxx, maxy = seed_gdf.total_bounds
margin = 0.08 * max(maxx - minx, maxy - miny)
ax.set_xlim(minx - margin, maxx + margin)
ax.set_ylim(miny - margin, maxy + margin)

cx.add_basemap(ax, source=cx.providers.OpenTopoMap, crs=refined_gdf.crs, zoom=10)

from matplotlib.lines import Line2D
from matplotlib.patches import Patch

ax.legend(
    handles=[
        Patch(facecolor="#27ae60", edgecolor="#1e8449", alpha=0.45, label=f"refined region ({props.get('n_basins')} basins ∩ bbox)"),
        Patch(facecolor="#f1c40f", edgecolor="#b7950b", alpha=0.25, label="selected basins (full extent)"),
        Line2D([0], [0], color="#c0392b", linestyle="--", label="seed bbox"),
        Line2D([0], [0], color="#2471a3", linestyle="--", label="Inn-line (hauptkamm)"),
    ],
    loc="lower right",
    framealpha=0.9,
)
ax.set_title(f"Komp. A — {REGION} (HydroBASINS L8, clipped, N of Inn)")
ax.set_xticks([])
ax.set_yticks([])
fig.tight_layout()

out = PROJECT_ROOT / "data" / "regions" / f"{REGION}_map.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print(f"saved {out.relative_to(PROJECT_ROOT)}")

## What to look for

- Does the refined polygon (green) hug the Karwendel / Rofan /
  Brandenberg / Steinberge ridges on the north side of the Inn?
- Does the seed bbox (red dashed) make sense as a region-of-interest,
  or is it cutting through something important?
- Is the Inn-line (blue dashed) running roughly along the Inn river on
  the basemap? (Visual sanity check of the LSQ calibration.)
- Are the selected-basins outlines (yellow) extending far outside the
  bbox? If yes, the clip-to-bbox step is doing real work.

Basemap tiles courtesy of OpenTopoMap (CC-BY-SA) on top of
OpenStreetMap data and SRTM elevation.